# 4. Inspeção das Propostas (Candidatos BAC)

## Objetivo
- Gerar imagens de inspeção: a mamografia em resolução original (ou `1024x1024` mapeada) com caixas numeradas.
- Extrair recortes (patches) das propostas na resolução original para exame detalhado.
- Preparar um registro estruturado (avaliação exploratória não-clínica) categorizando os candidatos como: **Plausível, Improvável ou Incerta**, incluindo justificativas.
- Mapear falhas comuns do baseline clássico (Frangi): artefatos, bordas, calcificações não vasculares, fragmentação.

In [1]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

In [2]:
def load_dicom_normalized(dcm_path):
    ds = pydicom.dcmread(dcm_path)
    image = ds.pixel_array.astype(np.float32)
    if getattr(ds, 'PhotometricInterpretation', 'UNKNOWN') == 'MONOCHROME1':
        image = np.max(image) - image
    norm = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
    return norm

## Geração de Patches e Mapa de Inspeção
O script a seguir itera sobre os candidatos gerados, plota a imagem inteira com as bounding boxes numeradas, recorta patches com um padding de contexto (ex: 50 pixels) para inspeção visual detalhada e cria um arquivo de revisão.

In [3]:
output_dir = Path('../../data/interim/inspection')
(output_dir / 'full_images').mkdir(parents=True, exist_ok=True)
(output_dir / 'patches').mkdir(parents=True, exist_ok=True)

manifest_path = Path('../../data/interim/inbreast_batch1_ml_manifest.csv')
df_batch = pd.read_csv(manifest_path)

try:
    df_cand = pd.read_csv('../../data/interim/baseline_candidates.csv')
except FileNotFoundError:
    print("Nenhum candidato salvo encontrado. Finalizando.")
    df_cand = pd.DataFrame()

reviews = []

if not df_cand.empty:
    for idx, row in df_batch.iterrows():
        img_id = str(row['image_id'])
        cands = df_cand[df_cand['image_id'] == int(img_id)]
        
        if cands.empty:
            continue
            
        # Carrega e redimensiona como feito na etapa 3
        img_norm = load_dicom_normalized(row['dcm_path'])
        img_resized = cv2.resize(img_norm, (1024, 1024), interpolation=cv2.INTER_AREA)
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        # Mostrar usando CLAHE apenas para facilitar a inspeção humana
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_vis = clahe.apply((img_resized * 255).astype(np.uint8))
        
        ax.imshow(img_vis, cmap='gray')
        ax.set_title(f"Inspeção Global: Imagem {img_id}")
        ax.axis('off')
        
        print(f"\n--- Extraindo patches para a Imagem {img_id} ---")
        
        for _, cand in cands.iterrows():
            rank = cand['rank']
            score = cand['score']
            bbox_str = cand['bbox'].strip("()").split(", ")
            min_r, min_c, max_r, max_c = map(int, bbox_str)
            
            # Caixa numerada na imagem completa
            rect = patches.Rectangle((min_c, min_r), max_c - min_c, max_r - min_r, 
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(min_c, max_r + 15, str(rank), color='red', fontsize=12, weight='bold')
            
            # Extrair patch em resolução original (ou do img_resized com zoom)
            pad = 50
            r1 = max(0, min_r - pad)
            c1 = max(0, min_c - pad)
            r2 = min(img_resized.shape[0], max_r + pad)
            c2 = min(img_resized.shape[1], max_c + pad)
            
            patch = img_vis[r1:r2, c1:c2]
            
            patch_path = output_dir / 'patches' / f"img_{img_id}_rank_{rank}.png"
            cv2.imwrite(str(patch_path), patch)
            
            # Adicionar entrada na tabela de revisão
            justificativa = "Estrutura alongada (alta excentricidade) detectada." if cand['elongation'] > 0.9 else "Estrutura brilhante de tamanho suspeito."
            reviews.append({
                'image_id': img_id,
                'rank': rank,
                'score': score,
                'patch_file': patch_path.name,
                'avaliacao': 'Pendente (Plausível/Improvável/Incerta)',
                'justificativa': justificativa,
                'falha_comum': '',
                'observacoes': ''
            })
            
        full_path = output_dir / 'full_images' / f"{img_id}_inspection.png"
        plt.savefig(full_path, bbox_inches='tight')
        plt.close(fig)

df_reviews = pd.DataFrame(reviews)
df_reviews.to_csv(output_dir / 'inspection_review_template.csv', index=False)
print("\nGerado template de revisão em data/interim/inspection/inspection_review_template.csv")
display(df_reviews.head())


--- Extraindo patches para a Imagem 51049276 ---

--- Extraindo patches para a Imagem 20587928 ---

--- Extraindo patches para a Imagem 53587104 ---

--- Extraindo patches para a Imagem 20588216 ---

Gerado template de revisão em data/interim/inspection/inspection_review_template.csv


,image_id,rank,score,patch_file,avaliacao,justificativa,falha_comum,observacoes
0,51049276,1,0.721436,img_51049276_rank_1.png,Pendente (Plausível/Improvável/Incerta),Estrutura alongada (alta excentricidade) detec...,,
1,20587928,1,0.665903,img_20587928_rank_1.png,Pendente (Plausível/Improvável/Incerta),Estrutura brilhante de tamanho suspeito.,,
2,53587104,1,0.694336,img_53587104_rank_1.png,Pendente (Plausível/Improvável/Incerta),Estrutura alongada (alta excentricidade) detec...,,
3,20588216,1,0.663471,img_20588216_rank_1.png,Pendente (Plausível/Improvável/Incerta),Estrutura brilhante de tamanho suspeito.,,
